**NOTE:** interactive notebook this was developed in — kept for reference. `ms5.py` (run via `run_ms5.sh`) is the script actually used to run MountainSort5 (an alternative sorter to Kilosort4, for comparison).

In [1]:
pip install mountainsort5

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import spikeinterface as si 
import spikeinterface.sorters as ss
from tempfile import TemporaryDirectory
import numpy as np
import spikeinterface as si
import spikeinterface.preprocessing as spre
import mountainsort5 as ms5
from mountainsort5.util import create_cached_recording
import os

In [3]:
ss.get_sorter_params_description("mountainsort5")

{'scheme': "Which sorting scheme to use: '1, '2', or '3'",
 'detect_threshold': 'Detection threshold - recommend to use the default',
 'detect_sign': 'Use -1 for detecting negative peaks, 1 for positive, 0 for both',
 'detect_time_radius_msec': 'Determines the minimum allowable time interval between detected spikes in the same spatial region',
 'snippet_T1': 'Number of samples before the peak to include in the snippet',
 'snippet_T2': 'Number of samples after the peak to include in the snippet',
 'npca_per_channel': 'Number of PCA features per channel in the initial dimension reduction step',
 'npca_per_subdivision': 'Number of PCA features to compute at each stage of clustering in the isosplit6 subdivision method',
 'snippet_mask_radius': 'Radius of the mask to apply to the extracted snippets',
 'scheme1_detect_channel_radius': 'Channel radius for excluding events that are too close in time in scheme 1',
 'scheme2_phase1_detect_channel_radius': 'Channel radius for excluding events tha

In [4]:
root_folder = '/nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_g_g0'
data_folder = f'{root_folder}/preprocessed_slice_550_1270s'
exp_folder = os.path.join(root_folder, 'ms5_test_550_1270s')
rec = si.load_extractor(data_folder)
rec

/tmp/ipykernel_139084/344763946.py:4: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  rec = si.load_extractor(data_folder)


BinaryFolderRecording: 355 channels - 30000.283521 Hz - 1 segments - 21,600,205 samples 
                       720.00s (12.00 minutes) - float16 dtype - 14.28 GiB

In [5]:
rec1 = spre.whiten(rec)
rec1

WhitenRecording: 355 channels - 30000.283521 Hz - 1 segments - 21,600,205 samples 
                 720.00s (12.00 minutes) - float16 dtype - 14.28 GiB

In [6]:
# check whitening 
traces = rec.get_traces(segment_index=0, start_frame = 0, end_frame = 10)
traces1 = rec1.get_traces(segment_index=0, start_frame = 0, end_frame = 10)
print(traces)
print(traces1)

[[-0.2827 -0.3323 -0.7188 ...  1.916   0.4255 -0.7373]
 [-0.825  -0.868  -0.7246 ...  2.193   2.555   1.669 ]
 [-1.047  -1.619   0.6606 ...  1.72    0.4573  1.452 ]
 ...
 [ 0.871   1.13   -0.3542 ...  1.056  -0.044   0.722 ]
 [-0.1859  0.6123 -0.8735 ...  0.8223  0.2358 -1.667 ]
 [-0.2673  0.      0.654  ... -0.576  -1.185   0.398 ]]
[[ 0.2073   0.12054 -0.4958  ...  1.64     0.5776  -0.694  ]
 [-0.4963  -0.4207  -0.3025  ...  1.612    2.422    1.434  ]
 [-1.307   -1.69     1.338   ...  1.361    0.4644   1.385  ]
 ...
 [ 0.679    0.7544  -1.3     ...  0.7744  -0.1844   0.5303 ]
 [-0.7217   0.3274  -1.85    ...  0.7344   0.2083  -1.669  ]
 [-0.645   -0.2773   0.6816  ... -0.51    -1.222    0.4153 ]]


In [8]:
sorting_parameters = ms5.Scheme2SortingParameters(phase1_detect_channel_radius=200, detect_channel_radius=100)
print(sorting_parameters)

Scheme2SortingParameters(phase1_detect_channel_radius=200, detect_channel_radius=100, phase1_detect_threshold=5.5, phase1_detect_time_radius_msec=1.5, detect_time_radius_msec=0.5, phase1_npca_per_channel=3, phase1_npca_per_subdivision=10, phase1_pairwise_merge_step=False, detect_sign=-1, detect_threshold=5.5, snippet_T1=20, snippet_T2=20, snippet_mask_radius=None, max_num_snippets_per_training_batch=200, classifier_npca=None, training_duration_sec=None, training_recording_sampling_mode='initial', classification_chunk_sec=None)


In [9]:
import logging

logging.basicConfig(
    level=logging.INFO,
    filename=os.path.join(exp_folder, "ms5_log.txt"),
    filemode="w",
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True, 
)

cached_recording = os.path.join(exp_folder, "cached_recording")
os.makedirs(cached_recording, exist_ok=True)

recording_cached = create_cached_recording(rec1, folder=cached_recording)

sorting = ms5.sorting_scheme2(
    recording=recording_cached,
    sorting_parameters=ms5.Scheme2SortingParameters(
        phase1_detect_channel_radius=200,
        detect_channel_radius=100,
    ),
)


write_binary_recording (no parallelization):   0%|          | 0/721 [00:00<?, ?it/s]

Using the full recording for training: 720.0000288386788 sec
Running phase 1 sorting
Number of channels: 355
Number of timepoints: 21600205
Sampling frequency: 30000.283520599252 Hz
Channel 0: [16.  0.]
Channel 1: [48.  0.]
Channel 2: [ 0. 20.]
Channel 3: [32. 20.]
Channel 4: [16. 40.]
Channel 5: [48. 40.]
Channel 6: [ 0. 60.]
Channel 7: [32. 60.]
Channel 8: [16. 80.]
Channel 9: [48. 80.]
Channel 10: [ 32. 100.]
Channel 11: [ 16. 120.]
Channel 12: [ 48. 120.]
Channel 13: [  0. 140.]
Channel 14: [ 32. 140.]
Channel 15: [ 16. 160.]
Channel 16: [ 48. 160.]
Channel 17: [  0. 180.]
Channel 18: [ 32. 180.]
Channel 19: [ 16. 200.]
Channel 20: [ 48. 200.]
Channel 21: [  0. 220.]
Channel 22: [ 32. 220.]
Channel 23: [ 16. 240.]
Channel 24: [ 48. 240.]
Channel 25: [  0. 260.]
Channel 26: [ 32. 260.]
Channel 27: [ 16. 280.]
Channel 28: [ 48. 280.]
Channel 29: [  0. 300.]
Channel 30: [ 32. 300.]
Channel 31: [ 16. 320.]
Channel 32: [ 48. 320.]
Channel 33: [  0. 340.]
Channel 34: [ 32. 340.]
Channel 

In [14]:
def smart_compute(analyzer_save, sorting):
    if os.path.exists(analyzer_save):
        print("analyzer folder exists, loading extensions...")
        sorting_analyzer = si.load_sorting_analyzer(folder = analyzer_save, format = 'binary_folder')
    else: 
        print("analyzer folder does not exist, creating new analyzer...")
        sorting_analyzer = si.create_sorting_analyzer(sorting = sorting, recording = rec, format = 'binary_folder', folder = analyzer_save)
    
    existing_exts = sorting_analyzer.get_loaded_extension_names()
    print("already available:", existing_exts)

    #print("computing sparsity...")
    #sparsity = si.compute_sparsity(sorting_analyzer, method = 'closest_channels', num_channels = 48)
    #print("finished computing sparsity")

    compute_dict = {
    'random_spikes': {}, #random spike selection across units 
    'waveforms':{},
    'principal_components':{'n_components': 6, 'whiten':False},
    'templates':{
    }
    }

    to_compute = {
        ext: params for ext, params in compute_dict.items() if ext not in existing_exts
    }
    print("need to compute:", list(to_compute.keys()))

    if len(to_compute) > 0: 
        print("computing...")
        sorting_analyzer.compute(to_compute, save = True, progress_bar = True, n_jobs = 8)
    
    return sorting_analyzer 

In [11]:
print(sorting)

NumpySorting: 12 units - 1 segments - 30.0kHz


In [12]:
analyzer_save = os.path.join(os.path.join(root_folder,'ms5_test'), 'si_analyzer')
sorting_analyzer = smart_compute(analyzer_save,  sorting) 

analyzer folder does not exist, creating new analyzer...


estimate_sparsity (no parallelization):   0%|          | 0/721 [00:00<?, ?it/s]

already available: []
need to compute: ['random_spikes', 'waveforms', 'principal_components']
computing...


compute_waveforms (workers: 8 processes):   0%|          | 0/721 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/12 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
sorting_analyzer.compute('templates', save = True, progress_bar = True, n_jobs = 8)

In [ ]:
from spikeinterface.exporters import export_to_phy
export_to_phy(sorting_analyzer=sorting_analyzer, 
              output_folder= os.path.join(exp_folder, 'export_to_phy'),
              copy_binary = False, 
              remove_if_exists = True,
              n_jobs = -1
              )

spike_amplitudes (workers: 8 processes):   0%|          | 0/721 [00:00<?, ?it/s]

extract PCs (workers: 8 processes):   0%|          | 0/721 [00:00<?, ?it/s]

Run:
phy template-gui  /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_g_g0/ms5_test_550_1270s/export_to_phy/params.py
